In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.history import HISTORY_FEATURES
from src.modeling import NUMERIC_FEATURES, make_features
from src.neural_data import build_neural_preprocessor

In [3]:
history_features = pd.read_parquet(
    project_root / "data/processed/lab03_history_features.parquet"
)

datasets = {}

for split_name in ["train", "validation"]:
    base = pd.read_parquet(
        project_root / f"data/processed/lab02_{split_name}.parquet"
    )

    enriched = base.merge(
        history_features,
        on="source_row",
        how="left",
        sort=False,
        validate="one_to_one",
    )

    assert enriched["source_row"].equals(base["source_row"])
    assert enriched[HISTORY_FEATURES].notna().all().all()

    datasets[split_name] = enriched

In [9]:
extra_features = tuple(HISTORY_FEATURES)

numeric_types = {
    column: np.float32
    for column in NUMERIC_FEATURES + HISTORY_FEATURES
}

train_features = make_features(
    datasets["train"],
    extra_numeric_features=extra_features,
).astype(numeric_types)

validation_features = make_features(
    datasets["validation"],
    extra_numeric_features=extra_features,
).astype(numeric_types)

preprocessor = build_neural_preprocessor()

X_train = preprocessor.fit_transform(train_features)
X_validation = preprocessor.transform(validation_features)

y_train = datasets["train"]["is_laundering"].to_numpy(
    dtype=np.float32
)
y_validation = datasets["validation"]["is_laundering"].to_numpy(
    dtype=np.float32
)

In [12]:
assert X_train.shape == (1_379_551, 44)
assert X_validation.shape == (965_524, 44)

assert y_train.sum() == 1_800
assert y_validation.sum() == 1_036

for name, matrix in [
    ("train", X_train),
    ("validation", X_validation),
]:
    assert matrix.dtype == np.float32
    assert np.isfinite(matrix).all()

    print(
        name,
        matrix.shape,
        matrix.dtype,
        f"{matrix.nbytes / 1024**2:.1f} MiB",
    )

train (1379551, 44) float32 231.6 MiB
validation (965524, 44) float32 162.1 MiB


In [17]:
cache_dir = project_root / "data/processed/lab05"
cache_dir.mkdir(parents=True, exist_ok=True)

for name, array in {
    "X_train": X_train,
    "X_validation": X_validation,
    "y_train": y_train,
    "y_validation": y_validation,
}.items():
    np.save(cache_dir / f"{name}.npy", array)

for split_name, frame in datasets.items():
    np.save(
        cache_dir / f"{split_name}_source_rows.npy",
        frame["source_row"].to_numpy(),
    )

joblib.dump(
    preprocessor,
    cache_dir / "preprocessor.joblib",
)

['/Users/sara/Projects/data-science-lab/data/processed/lab05/preprocessor.joblib']

### Neural input preparation

We log-scaled amounts and counts, standardized numerical features,
and one-hot encoded categories. Preprocessing was fitted on training only.

The inputs contain 44 float32 columns. Training uses about 231.6 MiB;
validation uses about 162.1 MiB. These figures exclude other process
memory and temporary arrays.

We saved the arrays, source-row alignment, and fitted preprocessor.

In [18]:
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.neural import NeuralBaseline, predict_scores

torch.set_num_threads(2)

cache_dir = project_root / "data/processed/lab05"

X_train = torch.from_numpy(np.load(cache_dir / "X_train.npy"))
y_train = torch.from_numpy(np.load(cache_dir / "y_train.npy"))

X_validation = torch.from_numpy(
    np.load(cache_dir / "X_validation.npy")
)
y_validation = np.load(cache_dir / "y_validation.npy")

print("Training:", X_train.shape, X_train.dtype, X_train.device)
print("Validation:", X_validation.shape)
print(f"Training laundering rate: {y_train.mean().item():.3%}")

Training: torch.Size([1379551, 44]) torch.float32 cpu
Validation: torch.Size([965524, 44])
Training laundering rate: 0.130%
